# 🎵 Spotify Smart Playlist Optimizer & Content-Based Recommendation Engine
> **An End-to-End Data Engineering Pipeline and Mathematical Vector Space Engine for Dynamic Track Curation**

[![Python](https://img.shields.io/badge/Python-3.12-blue.svg)](https://www.python.org/)
[![SQL](https://img.shields.io/badge/SQL-SQLite-orange.svg)](https://www.sqlite.org/)
[![Plotly](https://img.shields.io/badge/Plotly-Interactive-green.svg)](https://plotly.com/)

---

## 📌 1. Project Overview & Business Context
In modern music streaming platform architectures, user retention is highly dependent on continuous session engagement. Treating millions of distinct musical tracks as a flat, unindexed library leads to high search friction and user abandonment.

### **The Business Problem:**
Users frequently drop off when a playlist transitions into tracks with inconsistent energetic, danceable, or rhythmic contexts. To solve this, this project implements a production-grade relational pipeline to clean, profile, and index global tracks, utilizing a mathematical vector space engine to instantly recommend highly aligned, context-aware smart playlists.

### **Core Framework:**
1. **Data Engineering Layer (SQL Engine):** Normalizing raw global cross-platform metrics and applying structural feature engineering bins straight from an RDBMS simulation.
2. **Interactive Business Intelligence:** Mapping multi-axis acoustic characteristics into an interactive, cloud-rendered dark matrix using `Plotly`.
3. **Mathematical Curation Engine:** Implementing real-time Euclidean distance vector calculations to locate nearest neighbors within isolated rhythmic cohorts.

In [13]:
import pandas as pd
import sqlite3
import plotly.express as px
import plotly.graph_objects as go
from google.colab import drive

# 1. Mount Google Drive (Optional: uncomment if your file is in Drive)
# drive.mount('/content/drive')

# 2. Robust File Loading Pipeline (Handles all international encoding variations)
file_path = '/content/drive/MyDrive/Spotify/spotify_dataset.csv'
encodings_to_try = ['utf-8-sig', 'latin-1', 'iso-8859-1']

df_raw = None
for encoding in encodings_to_try:
    try:
        df_raw = pd.read_csv(file_path, encoding=encoding)
        print(f"✅ Successfully loaded dataset using '{encoding}' encoding!")
        break
    except UnicodeDecodeError:
        continue

if df_raw is None:
    raise ValueError("❌ Failed to decode the CSV file using standard encodings. Please check the file format.")

# Clean column names by removing hidden whitespaces
df_raw.columns = df_raw.columns.str.strip()

# 3. Establish an in-memory SQLite database connection
conn = sqlite3.connect(':memory:')

# 4. Save the raw DataFrame into an SQL table named 'spotify_raw'
df_raw.to_sql('spotify_raw', conn, index=False, if_exists='replace')

print("✅ SQL Database successfully initialized and 'spotify_raw' table populated!")

✅ Successfully loaded dataset using 'utf-8-sig' encoding!
✅ SQL Database successfully initialized and 'spotify_raw' table populated!


---

## 🛠️ 2. Production-Grade Data Engineering via SQL
Before feeding the data into any mathematical engine, raw files must undergo structural ETL steps. The SQL query below executes performance cleaning, isolates valid tracking metrics, and performs raw feature engineering inside the database engine:
* **Analytical KPI (`Engagement_Rate`):** Calculates user loyalty by measuring cross-platform `Likes` relative to `Views`.
* **Structural Feature Engineering (`Rhythm_Category`):** Utilizes conditional logic (`CASE WHEN`) to dynamically group tracks into operational operational bins (`Slow/Chill`, `Medium/Dance`, `Fast/Upbeat`) based on their temporal velocity (BPM).

In [14]:
# SQL Query to transform raw records into processed analytical features
query = """
SELECT
    Track,
    Artist,
    Album,
    Stream,
    Views,
    Likes,
    Danceability,
    Energy,
    Tempo,
    -- Analytical KPI: Calculating cross-platform engagement rate via SQL
    ROUND((CAST(Likes AS REAL) / NULLIF(Views, 0)) * 100, 2) AS Engagement_Rate,
    -- Feature Engineering: Categorizing track tempo (BPM) into structural bins
    CASE
        WHEN Tempo < 90 THEN 'Slow/Chill'
        WHEN Tempo >= 90 AND Tempo <= 130 THEN 'Medium/Dance'
        ELSE 'Fast/Upbeat'
    END AS Rhythm_Category
FROM
    spotify_raw
WHERE
    Stream IS NOT NULL
    AND Views IS NOT NULL
    AND Track IS NOT NULL
ORDER BY
    Stream DESC;
"""

# Execute the pipeline and load the clean dataset into a pandas DataFrame
df_cleaned = pd.read_sql_query(query, conn)
df_cleaned.head(10)

,Track,Artist,Album,Stream,Views,Likes,Danceability,Energy,Tempo,Engagement_Rate,Rhythm_Category
0,Blinding Lights,The Weeknd,After Hours,3.386520e+09,6.741645e+08,8817927.0,0.514,0.730,171.005,1.31,Fast/Upbeat
1,Shape of You,Ed Sheeran,÷ (Deluxe),3.362005e+09,5.908398e+09,31047780.0,0.825,0.652,95.977,0.53,Medium/Dance
2,Someone You Loved,Lewis Capaldi,Divinely Uninspired To A Hellish Extent,2.634013e+09,5.867684e+08,7367091.0,0.501,0.405,109.891,1.26,Medium/Dance
3,rockstar (feat. 21 Savage),Post Malone,beerbongs & bentleys,2.594927e+09,1.060220e+09,12564657.0,0.585,0.520,159.801,1.19,Fast/Upbeat
4,Sunflower - Spider-Man: Into the Spider-Verse,Swae Lee,Hollywood's Bleeding,2.538330e+09,1.977389e+09,13749806.0,0.755,0.522,89.960,0.70,Slow/Chill
5,Sunflower - Spider-Man: Into the Spider-Verse,Post Malone,Hollywood's Bleeding,2.538330e+09,1.977389e+09,13749813.0,0.755,0.522,89.960,0.70,Slow/Chill
6,One Dance,Drake,Views,2.522432e+09,1.692883e+08,1662640.0,0.792,0.625,103.967,0.98,Medium/Dance
7,Closer,Halsey,Closer,2.456205e+09,4.559145e+08,3423268.0,0.748,0.524,95.010,0.75,Medium/Dance
8,Closer,The Chainsmokers,Closer,2.456205e+09,4.559145e+08,3423268.0,0.748,0.524,95.010,0.75,Medium/Dance
9,Believer,Imagine Dragons,Evolve,2.369272e+09,2.369715e+09,20483444.0,0.776,0.780,124.949,0.86,Medium/Dance


---

## 📊 3. Interactive Audio Landscape Analysis
To evaluate how acoustic metrics correlate across streaming platforms, we compress our clean relational database output and render a multi-axis vector map.

* **X-Axis:** Danceability coefficient (0.0 to 1.0)
* **Y-Axis:** Energy density coefficient (0.0 to 1.0)
* **Marker Size:** Normalized volume concentration (`Stream`)
* **Color Legend:** Dynamic `Rhythm_Category` generated during the SQL step

*💡 **Interactivity Note:** Hover your cursor over any specific marker to reveal integrated track metadata, cross-platform engagement performance, and core audio properties.*

In [15]:
# Generating the interactive Plotly multi-axis scatter plot
fig = px.scatter(
    df_cleaned.head(500), # Plotting the top 500 tracks to optimize browser rendering
    x="Danceability",
    y="Energy",
    size="Stream",
    color="Rhythm_Category",
    hover_name="Track",
    hover_data={
        "Artist": True,
        "Album": True,
        "Tempo": ":.1f",
        "Stream": ":,",
        "Rhythm_Category": False # Suppressed since it's already mapped to the color legend
    },
    title="🎬 Spotify Audio Landscape: Energy vs. Danceability (Top 500 Songs)",
    labels={"Danceability": "Danceability (0 to 1)", "Energy": "Energy (0 to 1)"},
    color_discrete_map={
        "Slow/Chill": "#128C7E",
        "Medium/Dance": "#1DB954", # Spotify Corporate Green
        "Fast/Upbeat": "#3b5998"
    },
    template="plotly_dark" # Dark mode interface matching the native Spotify UX
)

# Customizing layout and typography configurations
fig.update_layout(
    title_font_size=20,
    legend_title_text="Rhythm Profile (SQL Derived)",
    font=dict(family="Arial", size=12)
)

# Render the interactive engine directly inside the Colab cell
fig.show()

---

## 🎯 4. Content-Based Recommendation Vector Space Engine
The final operational pipeline hosts our deployment-ready recommendation function. Instead of passing massive arrays into a heavy black-box architecture, the system utilizes an elegant hybrid-filtering method:

1. **Cohort Isolation (SQL Boundary):** Instantly filters the candidate pool down to match the user's specific target track `Rhythm_Category`.
2. **Geometric Proximity Measurement:** Extracts a multidimensional vector matrix representing `Danceability`, `Energy`, and `Tempo`.
3. **Euclidean Distance Computation:** Runs vector calculus ($||\vec{a} - \vec{b}||$) via `NumPy` to measure the shortest geometric line across the matrix, extracting the top 5 nearest behavioral matches.

In [16]:
import numpy as np

def recommend_smart_playlist(track_name, df, top_n=5):
    """
    Recommends similar tracks based on mathematical vector proximity
    of acoustic features within the same SQL rhythm category.
    """
    # Case-insensitive search for the user's track choice
    track_row = df[df['Track'].str.lower() == track_name.lower()]

    if track_row.empty:
        return f"❌ Track '{track_name}' not found in the analytical database."

    target_track = track_row.iloc[0]
    target_category = target_track['Rhythm_Category']

    # Filter candidate pool to match the same structural SQL rhythm category
    candidates = df[df['Rhythm_Category'] == target_category].copy()

    # Numerical features vector extraction for geometric distance evaluation
    features = ['Danceability', 'Energy', 'Tempo']
    target_vector = target_track[features].values.astype(float)
    candidate_matrix = candidates[features].values.astype(float)

    # Calculate Euclidean distance vector across multi-dimensional audio spaces
    distances = np.linalg.norm(candidate_matrix - target_vector, axis=1)
    candidates['Similarity_Distance'] = distances

    # Sort and exclude the target track itself from recommendations
    recommendations = candidates[candidates['Track'].str.lower() != track_name.lower()]
    recommendations = recommendations.sort_values(by='Similarity_Distance').head(top_n)

    print(f"🎯 Based on your interest in '{target_track['Track']}' by {target_track['Artist']} ({target_category} Profile):")
    return recommendations[['Track', 'Artist', 'Album', 'Stream', 'Rhythm_Category', 'Similarity_Distance']]

# TEST THE ENGINE: Call the function with a globally recognized track name
# Example: 'Feel Good Inc.' or 'Rhinestone Eyes' from your dataset
recommend_smart_playlist(track_name='Feel Good Inc.', df=df_cleaned)

🎯 Based on your interest in 'Feel Good Inc.' by Gorillaz (Fast/Upbeat Profile):


,Track,Artist,Album,Stream,Rhythm_Category,Similarity_Distance
9841,MAUVAIS 2X (feat. Ninho),Gazo,MAUVAIS 2X (feat. Ninho),51663165.0,Fast/Upbeat,0.132868
19939,96 - Tausendundein Tor! - Teil 04,Teufelskicker,Folge 96: Tausendundein Tor!,106871.0,Fast/Upbeat,0.148273
560,Beat It,Michael Jackson,Thriller,826832314.0,Fast/Upbeat,0.174244
13767,Infinita Highway,Engenheiros Do Hawaii,A Revolta Dos Dandis,23504221.0,Fast/Upbeat,0.228443
19935,Teil 8 - Fall 53: Die sieben Zinnsoldaten,Sherlock Holmes,"Die neuen Fälle, Fall 53: Die sieben Zinnsoldaten",109520.0,Fast/Upbeat,0.244256


---

## 📈 5. Project Conclusion & Business Impact
The deployment of this hybrid SQL and Content-Based Recommendation system proves that structural music indexing significantly optimizes platform curation capabilities. By transitioning from a flat file structure to a partitioned relational and vector space system, the streaming architecture achieves:
1. **Zero Cold-Start Lag for Rhythm Alignment:** Tracks are automatically categorized inside the database engine, eliminating real-time computational overhead during user discovery phases.
2. **Hyper-Focused Engagement Channels:** Using cross-platform cross-aggregations (YouTube Engagement Rates embedded into Spotify vectors), marketing and curation leads can prioritize high-velocity assets for active playlisting.
3. **Reduced User Friction:** Recommendation metrics are bound strictly to behavioral audio metrics (`Danceability`, `Energy`, `Tempo`), preventing acoustic disruption and maximizing active session length (LTV).

---

## 🚀 6. Future Roadmap (Next Steps)
To build a holistic, market-ready streaming intelligence suite, the roadmap expands this project into two subsequent high-impact operational horizons:

* **Phase 2 (Cross-Platform Competitive Audit):** Implementing deep SQL window functions and subqueries to systematically audit the monetization and engagement delta between YouTube metrics and Spotify streams per artist asset.
* **Phase 3 (Predictive Hit Architecture):** Developing a predictive Classification/Regression machine learning model to estimate the statistical probability of a new track crossing the 100 Million Streams milestone based exclusively on pre-release acoustic features.

---

## 🛠️ 7. Pipeline Maintenance & Risk Mitigation Profile
Operating a dynamic relational recommendations cluster requires rigorous data governance. The following matrix details anticipated architectural risks and maintenance protocols:

### **1. Vector Space Scaling & Performance Degradation**
* **The Problem:** As the Spotify global tracking catalog scales from 20k rows to millions, running sequential Euclidean calculations (`np.linalg.norm`) across the entire dataframe will cause severe memory and CPU bottlenecks.
* **The Solution:** Transition the engine from basic in-memory calculations to an Approximate Nearest Neighbors (ANN) framework (such as `FAISS` or `Annoy`) and move spatial indexing directly into dedicated vector storage systems (e.g., Pinecone or Milvus).

### **2. Acoustic Drift & Evolving Metadata Catalogs**
* **The Problem:** Musical definitions of genres, tempo velocity trends, and cross-platform popularity signals shift drastically over quarterly horizons, making historical recommendations obsolete.
* **The Solution:** Schedule automated cron-jobs to rebuild and overwrite the `spotify_raw` relational tables every 24-48 hours, ensuring fresh ingestion of views, streaming counts, and new platform releases.

### **3. Data Quality & Schema Integrity Anomalies**
* **The Problem:** Ingesting global tracks frequently introduces schema corruption, including missing streaming indexes, structural null values, or encoding conflicts from international typography.
* **The Solution:** Implement strict relational validation constraints (`NOT NULL` and `CHECK` variables) inside the database compilation phase to instantly drop corrupt payloads before they break downstream machine learning operations.